# 1. Purpose and scope

This notebook adds a **controlled frozen IndoBERTweet contextual-embedding extension** to the 42 existing TF-IDF configurations. It does not modify or rerun `experiment.ipynb`. The encoder is frozen; a fixed Logistic Regression linear head is evaluated under six sampling conditions and one class-weighted condition.

Final design: **42 existing + 6 IndoBERTweet sampling + 1 IndoBERTweet class-weighted = 49 configurations**. Each new condition uses seeds 42, 123, and 456 (21 new runs). No Transformer-generated labels and no end-to-end fine-tuning are used.

> Safety: every long-running phase is gated by an explicit `False` switch. Running all cells with the default switches does not download the checkpoint, extract embeddings, resample dense matrices, or train classifiers.

In [61]:
from pathlib import Path
import importlib.util
import json
import numpy as np
import pandas as pd

from src.indobertweet_extension import (
    EXTENSION_METHODS, SEEDS, audit_csv, combined_frame,
    environment_report, estimate_resources, extract_embeddings_once,
    generate_result_artifacts, merge_verified_results, paper_evidence_summary,
    prepare_output_paths, reconstruct_existing_split, reproduce_alignment,
    run_extension_experiments, save_split_indices, validate_dataset_files,
    validate_embedding_cache, validate_existing_results,
)

PROJECT_ROOT = Path(r"C:\Users\Asus\Documents\Kuliah\9\Akhir\Codes\analisis-sentimen-pertamax-oplosan")
assert (PROJECT_ROOT / 'experiment.ipynb').exists(), PROJECT_ROOT
RAW_ADJUDICATED_PATH = PROJECT_ROOT / 'data_kesepakatan_export.csv'
MODELING_PATH = PROJECT_ROOT / 'data' / 'processed' / 'preprocessed_dataset_kesepakatan.csv'
EXISTING_RESULTS_PATH = PROJECT_ROOT / 'results' / 'all_results.json'
MODEL_ID = 'indolem/indobertweet-base-uncased'
MAX_LENGTH = 128
EMBEDDING_DIMENSION = 768
EXTRACTION_BATCH_SIZE = 16  # conservative for the 4 GiB RTX 3050 Ti
LOGISTIC_MAX_ITER = 500     # matches src.models.create_logistic_regression
REPRO_SUBSET_SIZE = 100
REPRO_SUBSET_SEED = 42
REPRO_RTOL = 1e-6
REPRO_ATOL = 1e-7

RUN_EXISTING_42 = False
RUN_ALIGNMENT_VALIDATION = False
RUN_EMBEDDING_EXTRACTION = False
RUN_EXTENSION_EXPERIMENTS = False
RUN_MERGE_AND_VISUALS = True

assert RUN_EXISTING_42 is False, 'The historical 42 configurations must not be rerun here.'
print('Configuration loaded. All long-running switches are OFF.')

Configuration loaded. All long-running switches are OFF.


# 2. Environment and resource report

This fast preflight reports CPU, RAM, disk, PyTorch, CUDA, GPU, and VRAM when available. It does not install packages or initialize the model.

In [62]:
required_modules = ['torch', 'transformers', 'imblearn', 'sklearn', 'psutil']
module_status = {name: importlib.util.find_spec(name) is not None for name in required_modules}
resources = environment_report(expected_dimension=EMBEDDING_DIMENSION)
print('Module availability:', module_status)
print(json.dumps(resources, indent=2, ensure_ascii=False))
if not all(module_status.values()):
    print('STOP before extraction: install the missing modules and restart this kernel.')

Module availability: {'torch': True, 'transformers': True, 'imblearn': True, 'sklearn': True, 'psutil': True}
{
  "python": "3.11.5 (tags/v3.11.5:cce6ba9, Aug 24 2023, 14:38:34) [MSC v.1936 64 bit (AMD64)]",
  "platform": "Windows-10-10.0.26200-SP0",
  "cpu": "AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD",
  "logical_cores": 16,
  "expected_embedding_dimension": 768,
  "ram_total_gib": 15.405414581298828,
  "ram_available_gib": 3.1298294067382812,
  "disk_free_gib": 147.73374938964844,
  "torch_version": "2.7.1+cu118",
  "cuda_available": true,
  "gpu": "NVIDIA GeForce RTX 3050 Ti Laptop GPU",
  "vram_total_gib": 3.99951171875,
  "vram_free_gib": 3.171582031995058
}


If `torch` or `transformers` is missing, run the following manually in PowerShell **after the original visualization notebook has finished**, then restart this notebook's kernel:

```powershell
cd 'C:\Users\Asus\Documents\Kuliah\9\Akhir\Codes\analisis-sentimen-pertamax-oplosan'
.\.env\Scripts\Activate.ps1
python -m pip install torch transformers
```

Do not run two embedding extractors concurrently. The exact installed versions are recorded in cache metadata.

# 3. Configuration

Output paths are isolated under `results/indobertweet/`; `results/all_results.json`, existing predictions, figures, and `experiment.ipynb` are never overwritten.

In [63]:
paths = prepare_output_paths(PROJECT_ROOT)
for name, path in paths.items():
    print(f'{name:20s} -> {path}')
print('Seeds:', SEEDS)
print('New configurations:', EXTENSION_METHODS)

base                 -> C:\Users\Asus\Documents\Kuliah\9\Akhir\Codes\analisis-sentimen-pertamax-oplosan\results\indobertweet
cache                -> C:\Users\Asus\Documents\Kuliah\9\Akhir\Codes\analisis-sentimen-pertamax-oplosan\results\indobertweet\cache
predictions          -> C:\Users\Asus\Documents\Kuliah\9\Akhir\Codes\analisis-sentimen-pertamax-oplosan\results\indobertweet\indobertweet_predictions
reports              -> C:\Users\Asus\Documents\Kuliah\9\Akhir\Codes\analisis-sentimen-pertamax-oplosan\results\indobertweet\classification_reports
confusion_matrices   -> C:\Users\Asus\Documents\Kuliah\9\Akhir\Codes\analisis-sentimen-pertamax-oplosan\results\indobertweet\confusion_matrices
figures              -> C:\Users\Asus\Documents\Kuliah\9\Akhir\Codes\analisis-sentimen-pertamax-oplosan\results\indobertweet\figures
checkpoints          -> C:\Users\Asus\Documents\Kuliah\9\Akhir\Codes\analisis-sentimen-pertamax-oplosan\results\indobertweet\checkpoints
results              -> C:\Users

# 4. Dataset validation

The finalized adjudicated file (34,993 labels) and the exact modeling file (34,985 rows) are validated independently. This step reads evidence only.

In [64]:
raw_adjudicated, modeling, dataset_evidence = validate_dataset_files(
    RAW_ADJUDICATED_PATH, MODELING_PATH
)
print(json.dumps(dataset_evidence, indent=2, ensure_ascii=False))
assert len(raw_adjudicated) == 34_993
assert len(modeling) == 34_985
assert modeling['label'].value_counts().sort_index().to_dict() == {0: 27_350, 1: 6_649, 2: 986}

{
  "raw_adjudicated": {
    "path": "C:\\Users\\Asus\\Documents\\Kuliah\\9\\Akhir\\Codes\\analisis-sentimen-pertamax-oplosan\\data_kesepakatan_export.csv",
    "sha256": "5ac1647da3916450367b714056889a31c0eb2913bddca92a6e39e57263dc773a",
    "rows": 34993,
    "columns": [
      "clean_text",
      "label"
    ],
    "label_distribution": {
      "0": 27357,
      "1": 6650,
      "2": 986
    },
    "null_text": 0,
    "empty_text": 0,
    "unique_text": 34987,
    "duplicate_rows_beyond_first": 6
  },
  "modeling": {
    "path": "C:\\Users\\Asus\\Documents\\Kuliah\\9\\Akhir\\Codes\\analisis-sentimen-pertamax-oplosan\\data\\processed\\preprocessed_dataset_kesepakatan.csv",
    "sha256": "7fa752508ccf442cad37fc4ce64b376e7c2f1e42e20d5d8abf3b4e779f583619",
    "rows": 34985,
    "columns": [
      "clean_text",
      "label"
    ],
    "label_distribution": {
      "0": 27350,
      "1": 6649,
      "2": 986
    },
    "null_text": 0,
    "empty_text": 0,
    "unique_text": 33678,
    "

# 5. Text and label alignment

> **LONG-RUNNING VALIDATION — switch required.**

The raw adjudicated tweet rows are not joined by text. Instead, the recorded preprocessing pipeline is replayed while retaining a source-row index. Original text is accepted only if the reproduced 34,985 labels and processed texts match the modeling CSV exactly and in the same order. Eight removed rows are reported explicitly.

In [65]:
if RUN_ALIGNMENT_VALIDATION:
    aligned, alignment_evidence = reproduce_alignment(raw_adjudicated, modeling)
    print(json.dumps(alignment_evidence, indent=2, ensure_ascii=False))
else:
    print('SKIPPED: set RUN_ALIGNMENT_VALIDATION=True only when ready to replay Sastrawi preprocessing.')

SKIPPED: set RUN_ALIGNMENT_VALIDATION=True only when ready to replay Sastrawi preprocessing.


# 6. Existing split reconstruction

The existing runner uses `train_test_split(..., test_size=0.2, random_state=42)` without `stratify`. The same positional split is reconstructed; no group-aware change is introduced because that would invalidate direct comparison with the 42 historical configurations. Duplicate-text overlap is measured and retained as a limitation.

In [66]:
train_indices, test_indices, split_evidence = reconstruct_existing_split(modeling, seed=42)
assert split_evidence['train_rows'] == 27_988
assert split_evidence['test_rows'] == 6_997
assert split_evidence['train_distribution'] == {'0': 21_890, '1': 5_332, '2': 766}
assert split_evidence['test_distribution'] == {'0': 5_460, '1': 1_317, '2': 220}
save_split_indices(paths['split'], train_indices, test_indices, split_evidence)
print(json.dumps(split_evidence, indent=2, ensure_ascii=False))
resource_estimate = estimate_resources(
    split_evidence['train_distribution'], split_evidence['test_rows'], EMBEDDING_DIMENSION
)
print(json.dumps(resource_estimate, indent=2, ensure_ascii=False))

{
  "implementation": "sklearn.model_selection.train_test_split",
  "test_size": 0.2,
  "random_state": 42,
  "stratify": null,
  "matches_existing_code": true,
  "train_rows": 27988,
  "test_rows": 6997,
  "train_distribution": {
    "0": 21890,
    "1": 5332,
    "2": 766
  },
  "test_distribution": {
    "0": 5460,
    "1": 1317,
    "2": 220
  },
  "train_indices_sha256": "df0d613cbb0785bfdf446b87cf006356949e1841f86c52a1ca25a1e777f4aa00",
  "test_indices_sha256": "262a7ef52245577c16b85e2c5e217b79184e98d9bcc6844b2caed5e698faef57",
  "duplicate_text_overlap_unique": 246,
  "test_rows_with_text_seen_in_train": 351,
  "train_rows_with_text_seen_in_test": 738,
  "limitation": "The historical split is not stratified or group-aware; duplicate text overlaps train and test."
}
{
  "float_dtype": "float32",
  "bytes_per_embedding": 3072,
  "embedding_cache_mib": 102.4951171875,
  "largest_resampled_embedding_matrix_mib": 192.392578125,
  "expected_training_rows": {
    "No_Balancing": 27988,

# 7. IndoBERTweet tokenizer and encoder

Checkpoint: `indolem/indobertweet-base-uncased`. The tokenizer receives verified original tweets with only URL-to-`HTTPURL`, mention-to-`@USER`, and whitespace normalization. Stemming and stopword deletion are not applied. The frozen encoder runs in evaluation/inference mode. Last hidden states are pooled by an attention-mask-aware mean.

In [67]:
encoder_design = {
    'checkpoint': MODEL_ID,
    'frozen': True,
    'fine_tuning': False,
    'max_length': MAX_LENGTH,
    'pooling': 'attention-mask-aware mean pooling over last hidden state',
    'text': 'verified original text; URL/mention/whitespace normalization only',
}
print(json.dumps(encoder_design, indent=2))

{
  "checkpoint": "indolem/indobertweet-base-uncased",
  "frozen": true,
  "fine_tuning": false,
  "max_length": 128,
  "pooling": "attention-mask-aware mean pooling over last hidden state",
  "text": "verified original text; URL/mention/whitespace normalization only"
}


# 8. Cached embedding extraction

> **LONG-RUNNING — downloads/loads the model and extracts embeddings once.**

Before any full-dataset extraction is accepted, a fixed random subset of 100 rows is embedded twice with `model.eval()` and `torch.no_grad()`. The two outputs must have exactly equal shapes and satisfy `np.allclose(rtol=1e-6, atol=1e-7)`; maximum absolute difference and output hashes are recorded. If the test fails, diagnostic evidence is saved and full extraction does not start. The cache is reused only when this passed reproducibility proof, dataset hashes, row count, label distribution, split hashes, checkpoint, tokenizer, pooling, maximum length, dimension, package versions, device, batch size, array shapes, dtypes, and cache-file hashes all validate.

In [68]:
if RUN_EMBEDDING_EXTRACTION:
    if 'aligned' not in globals():
        raise RuntimeError('Run the verified alignment cell first; original text will not be guessed.')
    if not all(module_status.values()):
        raise RuntimeError('Install missing dependencies and restart the kernel first.')
    cache_metadata = extract_embeddings_once(
        aligned=aligned,
        train_indices=train_indices,
        test_indices=test_indices,
        cache_dir=paths['cache'],
        dataset_evidence=dataset_evidence,
        checkpoint=MODEL_ID,
        max_length=MAX_LENGTH,
        batch_size=EXTRACTION_BATCH_SIZE,
        expected_dimension=EMBEDDING_DIMENSION,
        reproducibility_subset_size=REPRO_SUBSET_SIZE,
        reproducibility_subset_seed=REPRO_SUBSET_SEED,
        reproducibility_rtol=REPRO_RTOL,
        reproducibility_atol=REPRO_ATOL,
    )
    print(json.dumps(cache_metadata, indent=2, ensure_ascii=False))
else:
    print('SKIPPED: embeddings have not been extracted.')

SKIPPED: embeddings have not been extracted.


# 9. Balancing methods

`No_Balancing`, `RandomOverSampler`, `SMOTE`, `RandomUnderSampler`, `SMOTEENN`, and `SMOTETomek` use the same default imbalanced-learn parameters and per-run `random_state` behavior as `src/balancing.py`. Labels remain one-dimensional for scikit-learn. Sampling touches only cached **training embeddings and training labels**—never raw text, token IDs, attention masks, test embeddings, or test labels.

In [69]:
method_validation = {
    'sampling_methods': EXTENSION_METHODS[:-1],
    'sampling_domain': 'dense frozen training embeddings only',
    'test_set_resampled': False,
    'reason': 'Synthetic/selection methods require numeric feature vectors; applying them to token IDs would create invalid token sequences.',
}
print(json.dumps(method_validation, indent=2))

{
  "sampling_methods": [
    "No_Balancing",
    "ROS",
    "SMOTE",
    "RUS",
    "SMOTEENN",
    "SMOTETomek"
  ],
  "sampling_domain": "dense frozen training embeddings only",
  "test_set_resampled": false,
  "reason": "Synthetic/selection methods require numeric feature vectors; applying them to token IDs would create invalid token sequences."
}


# 10. Linear classifier and class weighting

All seven configurations use `LogisticRegression(max_iter=500, random_state=seed)`. The `Class_Weighted` configuration alone adds `class_weight='balanced'` and uses the untouched training embeddings. This is the requested cost-sensitive baseline; focal loss is not evaluated.

In [70]:
linear_head_design = {
    'classifier': 'sklearn.linear_model.LogisticRegression',
    'max_iter': LOGISTIC_MAX_ITER,
    'random_state': 'current seed',
    'class_weight': {'sampling configurations': None, 'Class_Weighted': 'balanced'},
}
print(json.dumps(linear_head_design, indent=2))

{
  "classifier": "sklearn.linear_model.LogisticRegression",
  "max_iter": 500,
  "random_state": "current seed",
  "class_weight": {
    "sampling configurations": null,
    "Class_Weighted": "balanced"
  }
}


# 11. Three-seed experiment runner

> **LONG-RUNNING — seven configurations run sequentially.**

No two samplers run concurrently. If a method fails (especially SMOTEENN under memory pressure), its full traceback, stage, seed, and memory report are checkpointed and the remaining seeds for that configuration stop. No score is fabricated.

In [71]:
experiment_signature = {
    'design': 'frozen IndoBERTweet contextual embeddings + Logistic Regression',
    'model_id': MODEL_ID,
    'max_length': MAX_LENGTH,
    'embedding_dimension': EMBEDDING_DIMENSION,
    'batch_size': EXTRACTION_BATCH_SIZE,
    'seeds': SEEDS,
    'methods': EXTENSION_METHODS,
    'embedding_reproducibility': {
        'subset_size': REPRO_SUBSET_SIZE, 'subset_seed': REPRO_SUBSET_SEED,
        'passes': 2, 'rtol': REPRO_RTOL, 'atol': REPRO_ATOL,
    },
    'dataset_sha256': dataset_evidence['modeling']['sha256'],
    'raw_dataset_sha256': dataset_evidence['raw_adjudicated']['sha256'],
    'train_indices_sha256': split_evidence['train_indices_sha256'],
    'test_indices_sha256': split_evidence['test_indices_sha256'],
}
if RUN_EXTENSION_EXPERIMENTS:
    if not (paths['cache'] / 'metadata.json').exists():
        raise RuntimeError('Verified embedding cache does not exist.')
    extension_results = run_extension_experiments(
        cache_dir=paths['cache'],
        paths=paths,
        experiment_signature=experiment_signature,
        max_iter=LOGISTIC_MAX_ITER,
    )
else:
    print('SKIPPED: 21 new runs have not been started.')

SKIPPED: 21 new runs have not been started.


# 12. Metrics and prediction saving

Every completed seed saves accuracy; weighted precision, recall, and F1; macro-F1; per-class precision, recall, and F1 in label order 0/1/2; confusion matrix; classification report; sampling, training, and inference time; probabilities; predictions; true labels; split indices; sampler metadata; classifier parameters; and file hashes. Values are stored at full precision with `zero_division=0`.

In [72]:
metric_contract = [
    'accuracy', 'weighted precision', 'weighted recall', 'weighted F1', 'macro-F1',
    'negative precision/recall/F1', 'neutral precision/recall/F1',
    'positive precision/recall/F1', 'confusion matrix', 'classification report',
    'sampling/training/inference time', 'y_true/y_pred/probabilities',
]
print('Evidence contract:')
for item in metric_contract:
    print('-', item)

Evidence contract:
- accuracy
- weighted precision
- weighted recall
- weighted F1
- macro-F1
- negative precision/recall/F1
- neutral precision/recall/F1
- positive precision/recall/F1
- confusion matrix
- classification report
- sampling/training/inference time
- y_true/y_pred/probabilities


# 13. Resume and checkpoint logic

Each seed writes a checkpoint immediately. A run is skipped on resume only when its status is `complete`, its experiment-signature hash matches, and its prediction archive exists. Failed or mismatched runs are not treated as complete. Population standard deviation uses `ddof=0`, matching the historical pipeline.

In [73]:
checkpoint_files = sorted(paths['checkpoints'].glob('*.json'))
print(f'Current checkpoint files: {len(checkpoint_files)}')
for checkpoint in checkpoint_files:
    record = json.loads(checkpoint.read_text(encoding='utf-8'))
    print(checkpoint.name, record.get('status'), record.get('method'), record.get('seed'))

Current checkpoint files: 21
Class_Weighted_seed123.json complete Class_Weighted 123
Class_Weighted_seed42.json complete Class_Weighted 42
Class_Weighted_seed456.json complete Class_Weighted 456
No_Balancing_seed123.json complete No_Balancing 123
No_Balancing_seed42.json complete No_Balancing 42
No_Balancing_seed456.json complete No_Balancing 456
ROS_seed123.json complete ROS 123
ROS_seed42.json complete ROS 42
ROS_seed456.json complete ROS 456
RUS_seed123.json complete RUS 123
RUS_seed42.json complete RUS 42
RUS_seed456.json complete RUS 456
SMOTE_seed123.json complete SMOTE 123
SMOTE_seed42.json complete SMOTE 42
SMOTE_seed456.json complete SMOTE 456
SMOTEENN_seed123.json complete SMOTEENN 123
SMOTEENN_seed42.json complete SMOTEENN 42
SMOTEENN_seed456.json complete SMOTEENN 456
SMOTETomek_seed123.json complete SMOTETomek 123
SMOTETomek_seed42.json complete SMOTETomek 42
SMOTETomek_seed456.json complete SMOTETomek 456


# 14. Existing-result validation

The historical JSON must contain exactly 42 configurations, three runs per configuration, and complete mean/std/run evidence for weighted F1, macro-F1, and each class F1. It is loaded read-only and never overwritten.

In [74]:
existing_validation = validate_existing_results(EXISTING_RESULTS_PATH)
print({key: value for key, value in existing_validation.items() if key != 'payload'})
assert existing_validation['configuration_count'] == 42
assert existing_validation['runs_per_configuration'] == 3

{'path': 'C:\\Users\\Asus\\Documents\\Kuliah\\9\\Akhir\\Codes\\analisis-sentimen-pertamax-oplosan\\results\\all_results.json', 'sha256': 'ed9b3366fc4641e1c384441ffcfb78dd81342787ed25db44e73d0f8ea5c65573', 'configuration_count': 42, 'runs_per_configuration': 3, 'required_metrics': ['accuracy', 'f1', 'f1_macro', 'f1_negative', 'f1_neutral', 'f1_positive', 'precision', 'recall']}


# 15. Merge into 49 configurations

The merge is allowed only after all seven new configurations have exactly three completed runs. Rankings are recalculated from the combined file; outcomes are not selected to preserve a prior conclusion.

In [75]:
if RUN_MERGE_AND_VISUALS:
    extension_results = json.loads(paths['results'].read_text(encoding='utf-8'))
    combined_results = merge_verified_results(
        existing_validation, extension_results, paths['combined']
    )
    assert combined_results['configuration_count'] == 49
    print('Merged configuration count:', combined_results['configuration_count'])
else:
    print('SKIPPED: merge waits for 7 x 3 verified new runs.')

Merged configuration count: 49


# 16. Rankings

Weighted F1 is the primary ranking, with macro-F1 and negative/neutral/positive F1 reported as complementary evidence. No significance, equivalence, indistinguishability, universal-superiority, or strong-minority-detection claim is inferred from weighted F1 alone.

In [76]:
if RUN_MERGE_AND_VISUALS:
    ranking_table = combined_frame(combined_results).sort_values(
        ['f1_mean', 'f1_macro_mean'], ascending=False
    )
    display(ranking_table[['family', 'configuration', 'f1_mean', 'f1_std', 'f1_macro_mean', 'f1_negative_mean', 'f1_neutral_mean', 'f1_positive_mean']])
    evidence_summary = paper_evidence_summary(combined_results)
    print(json.dumps(evidence_summary, indent=2, ensure_ascii=False))
else:
    print('SKIPPED: no combined 49-result evidence yet.')

,family,configuration,f1_mean,f1_std,f1_macro_mean,f1_negative_mean,f1_neutral_mean,f1_positive_mean
12,TF-IDF,Random Forest + ROS,0.761245,0.001097,0.509302,0.867640,0.404610,0.255654
5,TF-IDF,Random Forest + No_Balancing,0.755999,0.001118,0.485968,0.878438,0.333613,0.245854
42,Frozen IndoBERTweet,IndoBERTweet + No_Balancing,0.755007,0.000000,0.494661,0.881196,0.308880,0.293907
40,TF-IDF,Random Forest + SMOTETomek,0.751987,0.002065,0.502068,0.858019,0.395868,0.252315
19,TF-IDF,Random Forest + SMOTE,0.751326,0.000656,0.501614,0.857742,0.393253,0.253846
6,TF-IDF,Logistic Regression + No_Balancing,0.750337,0.000000,0.474621,0.882333,0.285714,0.255814
1,TF-IDF,MLP Advanced + No_Balancing,0.746366,0.003696,0.467133,0.882423,0.264471,0.254504
4,TF-IDF,SVM + No_Balancing,0.742317,0.000000,0.453068,0.882071,0.248786,0.228346
2,TF-IDF,MLP Keras Tuner + No_Balancing,0.741044,0.007076,0.454840,0.880924,0.245077,0.238518
3,TF-IDF,Naive Bayes + No_Balancing,0.739603,0.000000,0.452471,0.880007,0.241692,0.235714


{
  "best_f1": {
    "configuration": "Random Forest + ROS",
    "mean": 0.7612450332422037,
    "std": 0.001097449305626085
  },
  "best_f1_macro": {
    "configuration": "Random Forest + ROS",
    "mean": 0.5093015133699406,
    "std": 0.000582374161572058
  },
  "best_f1_negative": {
    "configuration": "MLP Advanced + No_Balancing",
    "mean": 0.882422565612876,
    "std": 0.000465088965022333
  },
  "best_f1_neutral": {
    "configuration": "Random Forest + ROS",
    "mean": 0.4046103358180007,
    "std": 0.003498687631903933
  },
  "best_f1_positive": {
    "configuration": "IndoBERTweet + No_Balancing",
    "mean": 0.2939068100358423,
    "std": 0.0
  },
  "best_indobertweet": {
    "configuration": "IndoBERTweet + No_Balancing",
    "weighted_f1": 0.7550074909467851,
    "macro_f1": 0.49466118549655613
  },
  "best_tfidf": {
    "configuration": "Random Forest + ROS",
    "weighted_f1": 0.7612450332422037,
    "macro_f1": 0.5093015133699406
  },
  "weighted_f1_difference_perc

# 17. Visualizations

> **POST-EXPERIMENT OUTPUT — switch required.**

This phase creates separate 7-row and 49-row tables, weighted/macro rankings, the best IndoBERTweet confusion matrix aggregated over three seeds, best-overall saved confusion evidence, the key classwise comparison, seed stability, and new-run timing. Historical per-seed predictions were not stored, so an existing best-overall TF-IDF confusion image is copied and explicitly marked as historical evidence rather than recomputed.

In [77]:
if RUN_MERGE_AND_VISUALS:
    artifact_manifest = generate_result_artifacts(
        combined_results, extension_results, paths, PROJECT_ROOT
    )
    print(json.dumps(artifact_manifest, indent=2, ensure_ascii=False))
else:
    print('SKIPPED: visualizations require complete combined evidence.')

{
  "tables": {
    "indobertweet_7_results.csv": "C:\\Users\\Asus\\Documents\\Kuliah\\9\\Akhir\\Codes\\analisis-sentimen-pertamax-oplosan\\results\\indobertweet\\figures\\indobertweet_7_results.csv",
    "all_49_weighted_f1.csv": "C:\\Users\\Asus\\Documents\\Kuliah\\9\\Akhir\\Codes\\analisis-sentimen-pertamax-oplosan\\results\\indobertweet\\figures\\all_49_weighted_f1.csv",
    "all_49_macro_f1.csv": "C:\\Users\\Asus\\Documents\\Kuliah\\9\\Akhir\\Codes\\analisis-sentimen-pertamax-oplosan\\results\\indobertweet\\figures\\all_49_macro_f1.csv",
    "all_49_classwise_f1.csv": "C:\\Users\\Asus\\Documents\\Kuliah\\9\\Akhir\\Codes\\analisis-sentimen-pertamax-oplosan\\results\\indobertweet\\figures\\all_49_classwise_f1.csv"
  },
  "figures": {
    "ranked_weighted_f1_49.png": "C:\\Users\\Asus\\Documents\\Kuliah\\9\\Akhir\\Codes\\analisis-sentimen-pertamax-oplosan\\results\\indobertweet\\figures\\ranked_weighted_f1_49.png",
    "ranked_macro_f1_49.png": "C:\\Users\\Asus\\Documents\\Kuliah\\9\\

# 18. Paper-ready result tables

Tables are generated only from `combined_results_49.json`. In the manuscript, introduce every table and figure in prose before it appears. Report mean ± population SD over three seeds and retain classwise F1 beside weighted and macro-F1.

In [78]:
if RUN_MERGE_AND_VISUALS:
    paper_columns = [
        'configuration', 'f1_mean', 'f1_std', 'f1_macro_mean', 'f1_macro_std',
        'f1_negative_mean', 'f1_neutral_mean', 'f1_positive_mean',
    ]
    indo_table = ranking_table[ranking_table['family'] == 'Frozen IndoBERTweet'][paper_columns]
    display(indo_table)
else:
    print('SKIPPED: paper-ready values must not be invented before the runs finish.')

,configuration,f1_mean,f1_std,f1_macro_mean,f1_macro_std,f1_negative_mean,f1_neutral_mean,f1_positive_mean
42,IndoBERTweet + No_Balancing,0.755007,0.000000,0.494661,0.000000,0.881196,0.308880,0.293907
48,IndoBERTweet + Class_Weighted,0.612232,0.000000,0.400029,0.000000,0.684677,0.393856,0.121554
43,IndoBERTweet + ROS,0.611801,0.002336,0.398802,0.002236,0.685102,0.389819,0.121486
44,IndoBERTweet + SMOTE,0.601514,0.001366,0.391233,0.001535,0.674508,0.379360,0.119830
47,IndoBERTweet + SMOTETomek,0.601093,0.002096,0.390991,0.001888,0.673899,0.379730,0.119345
45,IndoBERTweet + RUS,0.513734,0.008285,0.337167,0.005570,0.572436,0.339562,0.099503
46,IndoBERTweet + SMOTEENN,0.194716,0.003721,0.195926,0.002133,0.167177,0.325535,0.095066


# 19. Practical Design Recommendations

This section must be written only after verified results exist. It will compare the highest-performing configuration, RF+ROS, RF without balancing, the best frozen-IndoBERTweet configuration, sampling risks, and the class-weighted alternative. Every recommendation is limited to this Pertamax dataset, this historical split, and these evaluated configurations.

In [79]:
if RUN_MERGE_AND_VISUALS:
    print('Evidence boundary for the subsection:')
    print(json.dumps(evidence_summary, indent=2, ensure_ascii=False))
    print('Write recommendations conditionally from these verified values; do not generalize beyond the evaluated dataset/split/configurations.')
else:
    print('PENDING: practical recommendations wait for verified 49-configuration results.')

Evidence boundary for the subsection:
{
  "best_f1": {
    "configuration": "Random Forest + ROS",
    "mean": 0.7612450332422037,
    "std": 0.001097449305626085
  },
  "best_f1_macro": {
    "configuration": "Random Forest + ROS",
    "mean": 0.5093015133699406,
    "std": 0.000582374161572058
  },
  "best_f1_negative": {
    "configuration": "MLP Advanced + No_Balancing",
    "mean": 0.882422565612876,
    "std": 0.000465088965022333
  },
  "best_f1_neutral": {
    "configuration": "Random Forest + ROS",
    "mean": 0.4046103358180007,
    "std": 0.003498687631903933
  },
  "best_f1_positive": {
    "configuration": "IndoBERTweet + No_Balancing",
    "mean": 0.2939068100358423,
    "std": 0.0
  },
  "best_indobertweet": {
    "configuration": "IndoBERTweet + No_Balancing",
    "weighted_f1": 0.7550074909467851,
    "macro_f1": 0.49466118549655613
  },
  "best_tfidf": {
    "configuration": "Random Forest + ROS",
    "weighted_f1": 0.7612450332422037,
    "macro_f1": 0.50930151336994

# 20. Reviewer-response evidence

After completion, the response to Reviewer B may factually state that the revision adds: (1) Practical Design Recommendations, (2) a frozen IndoBERTweet contextual-embedding comparison with a supervised linear head, (3) a class-weighted cost-sensitive baseline, and (4) weighted, macro, and classwise F1 interpretation. It must not claim focal loss, full fine-tuning, statistical significance/equivalence, or resolved duplicate overlap. Group-aware splitting remains future work.

In [80]:
print('PRE-TRAINING STOP POINT')
print('Notebook implementation is ready; long cells remain disabled.')
print('Next sequence after the original notebook finishes: alignment -> embedding extraction once -> seven configurations -> merge/visuals.')

PRE-TRAINING STOP POINT
Notebook implementation is ready; long cells remain disabled.
Next sequence after the original notebook finishes: alignment -> embedding extraction once -> seven configurations -> merge/visuals.
